In [1]:
import numpy as np
import sklearn.base as skb
import sklearn.feature_selection as skf
import sklearn.utils.validation as skv
from plotly.io import show
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import validate_data

from skfolio.datasets import load_sp500_dataset
from skfolio.model_selection import (
    WalkForward,
    cross_val_predict,
)
from skfolio.optimization import EqualWeighted
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()
X = prices_to_returns(prices)

In [2]:
volumes_usd = np.random.rand(*X.shape) * 1e6

In [3]:
class VolumePreSelection(skf.SelectorMixin, skb.BaseEstimator):
    to_keep_: np.ndarray

    def __init__(self, pct_to_keep: float = 0.5):
        self.pct_to_keep = pct_to_keep

    def fit(self, X, y=None, volumes=None):
        # Validate and convert X to a NumPy array
        X = validate_data(self, X)

        # Check parameters
        if not 0 < self.pct_to_keep <= 1:
            raise ValueError(
                "`pct_to_keep` must be between 0 and 1"
            )

        # Validate and convert volumes to a NumPy array
        volumes = skv.check_array(
            volumes,
            accept_sparse=False,
            ensure_2d=False,
            dtype=[np.float64, np.float32],
            order="C",
            copy=False,
            input_name="volumes",
        )
        if volumes.shape != X.shape:
            raise ValueError(
                f"Volume data {volumes.shape} must have the same dimensions as X {X.shape}"
            )

        n_assets = X.shape[1]
        mean_volumes = volumes.mean(axis=0)

        # Select the top `pct_to_keep` assets with the highest average volumes
        n_to_keep = max(1, int(round(self.pct_to_keep * n_assets)))
        selected_idx = np.argsort(mean_volumes)[-n_to_keep:]

        # Performance tip: `argpartition` could be used here for better efficiency
        # (O(n log(n)) vs O(n)).
        self.to_keep_ = np.isin(np.arange(n_assets), selected_idx)
        return self

    def _get_support_mask(self):
        skv.check_is_fitted(self)
        return self.to_keep_

In [4]:
set_config(enable_metadata_routing=True, transform_output="pandas")

model = Pipeline(
    [
        (
            "pre_selection",
            VolumePreSelection(pct_to_keep=0.3).set_fit_request(
                volumes=True
            ),
        ),
        ("optimization", EqualWeighted()),
    ]
)

In [5]:
cv = WalkForward(test_size=3, train_size=6, freq="WOM-3FRI")

pred = cross_val_predict(model, X, cv=cv, params={"volumes": volumes_usd})

In [6]:
pred.composition

,EqualWeighted,EqualWeighted_1,EqualWeighted_2,EqualWeighted_3,EqualWeighted_4,EqualWeighted_5,EqualWeighted_6,EqualWeighted_7,EqualWeighted_8,EqualWeighted_9,...,EqualWeighted_119,EqualWeighted_120,EqualWeighted_121,EqualWeighted_122,EqualWeighted_123,EqualWeighted_124,EqualWeighted_125,EqualWeighted_126,EqualWeighted_127,EqualWeighted_128
asset,,,,,,,,,,,,,,,,,,,,,
JNJ,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.166667,0.166667,0.166667,...,0.000000,0.000000,0.166667,0.166667,0.166667,0.166667,0.000000,0.000000,0.000000,0.000000
JPM,0.166667,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.166667,0.000000,...,0.000000,0.000000,0.166667,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.166667
MRK,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PEP,0.166667,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.166667,0.166667,0.166667,...,0.000000,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
UNH,0.166667,0.000000,0.000000,0.166667,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,...,0.166667,0.166667,0.166667,0.000000,0.000000,0.166667,0.166667,0.166667,0.000000,0.000000
XOM,0.166667,0.166667,0.000000,0.000000,0.000000,0.166667,0.166667,0.166667,0.166667,0.000000,...,0.000000,0.000000,0.166667,0.000000,0.000000,0.166667,0.166667,0.000000,0.000000,0.000000
AAPL,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.166667,0.166667,0.000000,0.166667,...,0.166667,0.000000,0.000000,0.000000,0.166667,0.166667,0.166667,0.166667,0.166667,0.000000
KO,0.000000,0.166667,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.166667,0.166667,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000
LLY,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.166667,0.000000,0.166667,0.166667,0.166667,0.166667


In [7]:
pred.weights_per_observation.tail()

,JNJ,JPM,MRK,PEP,UNH,XOM,AAPL,KO,LLY,PG,AMD,BAC,MSFT,CVX,WMT,BBY,GE,HD,PFE,RRC
2022-10-14,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.166667,0.166667,0.0,0.166667,0.0,0.166667
2022-10-17,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.166667,0.166667,0.0,0.166667,0.0,0.166667
2022-10-18,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.166667,0.166667,0.0,0.166667,0.0,0.166667
2022-10-19,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.166667,0.166667,0.0,0.166667,0.0,0.166667
2022-10-20,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.166667,0.166667,0.0,0.166667,0.0,0.166667


In [8]:
fig = pred.plot_composition()
show(fig)

In [9]:
pred.plot_cumulative_returns()